# PAN tutorial notebook

This notebook teaches the **Phase Accumulator Network (PAN)** from first principles and then walks through a compact implementation you can run and modify.

It is built directly from your `pan.py` and README. The goal is not just to show code, but to make the architecture legible:

1. what problem PAN is solving,
2. why phase arithmetic is the core inductive bias,
3. how each module works,
4. how training and analysis fit together,
5. how to interpret the learned frequencies and ablations.


## The short intuition

A standard network treats multiplication and addition as primitive. PAN treats **phase rotation** as primitive.

For modular arithmetic, that gives you a very natural recipe:

- encode each integer as one or more angles on the unit circle,
- combine those angles with learned phase mixing,
- detect alignment with learned reference phases,
- decode the resulting phase-selective activations into logits.

So instead of hoping gradient descent discovers a Fourier-like circuit inside a generic architecture, PAN starts with a circuit family that already lives on the circle.


In [ ]:
import math
import time
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

PHASE_SCALE = 65536
PHASE_SCALE_F = 65536.0
TWO_PI = 2.0 * math.pi

print("device:", DEVICE)


## The task: modular addition

We start with the cleanest setting:

\[
(a, b) \mapsto (a+b) \bmod p
\]

for a prime `p`.

This is useful because the structure is extremely crisp. The additive group over a prime field has a natural Fourier basis, so if the architecture really is learning a phase-native solution, this task should expose it.


In [ ]:
def make_modular_dataset(p: int, train_frac: float = 0.4, seed: int = 42):
    """Return train/val tensors for all (a, b, (a+b) mod p) triples."""
    rng = np.random.default_rng(seed)
    pairs = np.array(
        [(a, b, (a + b) % p) for a in range(p) for b in range(p)],
        dtype=np.int64,
    )
    perm = rng.permutation(len(pairs))
    pairs = pairs[perm]

    n_train = int(train_frac * len(pairs))
    train = torch.tensor(pairs[:n_train], device=DEVICE)
    val = torch.tensor(pairs[n_train:], device=DEVICE)

    return train[:, :2], train[:, 2], val[:, :2], val[:, 2]

p = 11
train_x, train_y, val_x, val_y = make_modular_dataset(p)
print("train:", train_x.shape, train_y.shape)
print("val:", val_x.shape, val_y.shape)
print("sample pairs:", list(zip(train_x[:5].tolist(), train_y[:5].tolist())))


## Why phases help

If you encode an integer `a` using an angle

\[
\phi_k(a) = a \cdot \omega_k \pmod{2\pi}
\]

then modular addition becomes angle addition:

\[
\phi_k(a+b \bmod p) = \phi_k(a) + \phi_k(b) \pmod{2\pi}
\]

That is the whole game.

The transformer story is: a generic architecture eventually discovers sinusoidal structure.  
The PAN story is: make sinusoidal structure the native representation.


In [ ]:
def plot_theoretical_phase_encodings(p: int = 11, k_freqs: int = 4):
    xs = np.arange(p)
    fig, axes = plt.subplots(1, k_freqs, figsize=(4 * k_freqs, 3), sharey=True)
    if k_freqs == 1:
        axes = [axes]

    for k, ax in enumerate(axes, start=1):
        freq = k * TWO_PI / p
        phases = (xs * freq) % TWO_PI
        ax.plot(xs, phases, marker="o")
        ax.set_title(f"k={k}, freq={freq:.3f}")
        ax.set_xlabel("token")
        ax.set_ylabel("phase (rad)")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_theoretical_phase_encodings(p=11, k_freqs=4)


## Step 1: PhaseEncoder

This module turns a token into `K` phases. In your implementation, the frequencies are **learned**, but initialized to the natural Fourier basis:

\[
\omega_k = (k+1) \cdot \frac{2\pi}{p}
\]

That means the model starts near a meaningful basis instead of a random one.


In [ ]:
class PhaseEncoder(nn.Module):
    def __init__(self, p: int, k_freqs: int):
        super().__init__()
        self.p = p
        self.k_freqs = k_freqs

        init_freqs = torch.tensor(
            [(k + 1) * TWO_PI / p for k in range(k_freqs)],
            dtype=torch.float32,
        )
        self.freq = nn.Parameter(init_freqs)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        a = tokens.float().unsqueeze(-1)      # (batch, 1)
        f = self.freq.unsqueeze(0)            # (1, K)
        phases = (a * f) % TWO_PI             # (batch, K)
        return phases

encoder = PhaseEncoder(p=11, k_freqs=4).to(DEVICE)
demo_tokens = torch.tensor([0, 1, 2, 3, 10], device=DEVICE)
encoder(demo_tokens)


## Step 2: PhaseMixingLayer

Once you have phase vectors for `a` and `b`, PAN concatenates them and mixes them:

\[
\phi_{out}[j] = \left(\sum_i W[j,i]\,\phi_{in}[i]\right) \bmod 2\pi
\]

This is the learned phase-composition step. It is where the model decides how to combine the circular representations of the two inputs.


In [ ]:
class PhaseMixingLayer(nn.Module):
    def __init__(self, n_in: int, n_out: int):
        super().__init__()
        w_init = torch.randn(n_out, n_in) * 0.1 + (1.0 / n_in)
        self.weight = nn.Parameter(w_init)

    def forward(self, phases_in: torch.Tensor) -> torch.Tensor:
        return F.linear(phases_in, self.weight) % TWO_PI


## Step 3: PhaseGate

PAN does not use ReLU or GELU. It uses **phase alignment**.

For each channel:

\[
gate_j = \frac{1 + \cos(\phi_j - \phi^{ref}_j)}{2}
\]

That means the gate fires when the incoming phase is aligned with a learned reference phase.

Your current code also includes an important stability fix: `ref_phase` is wrapped back into `[0, 2π)` inside `forward()`. That prevents optimizer drift from turning the phase parameter into a numerically awkward real-valued proxy for an angle.


In [ ]:
class PhaseGate(nn.Module):
    def __init__(self, n_phases: int):
        super().__init__()
        self.ref_phase = nn.Parameter(torch.rand(n_phases) * TWO_PI)

    def forward(self, phases: torch.Tensor) -> torch.Tensor:
        ref = torch.remainder(self.ref_phase, TWO_PI)
        phase_diff = phases - ref.unsqueeze(0)
        return (1.0 + torch.cos(phase_diff)) / 2.0


## Step 4: Assemble the full PAN

The full path is:

1. encode `a`,
2. encode `b`,
3. concatenate the phase banks,
4. mix them,
5. gate by phase alignment,
6. decode the resulting activations into `p` logits.


In [ ]:
class PhaseAccumulatorNetwork(nn.Module):
    def __init__(self, p: int, k_freqs: int = 5):
        super().__init__()
        self.p = p
        self.k_freqs = k_freqs

        self.encoder_a = PhaseEncoder(p, k_freqs)
        self.encoder_b = PhaseEncoder(p, k_freqs)
        self.phase_mix = PhaseMixingLayer(2 * k_freqs, k_freqs)
        self.phase_gate = PhaseGate(k_freqs)
        self.decoder = nn.Linear(k_freqs, p, bias=True)

        nn.init.normal_(self.decoder.weight, std=0.02)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        a = inputs[:, 0]
        b = inputs[:, 1]

        phi_a = self.encoder_a(a)
        phi_b = self.encoder_b(b)
        phi_mixed = self.phase_mix(torch.cat([phi_a, phi_b], dim=-1))
        gates = self.phase_gate(phi_mixed)
        return self.decoder(gates)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_learned_frequencies(self) -> dict:
        freqs_a_raw = self.encoder_a.freq.detach().cpu().numpy()
        freqs_b_raw = self.encoder_b.freq.detach().cpu().numpy()
        freqs_a = freqs_a_raw % TWO_PI
        freqs_b = freqs_b_raw % TWO_PI
        theoretical = np.array([(k + 1) * TWO_PI / self.p for k in range(self.k_freqs)])

        def _angle_err(learned, theory):
            diff = np.abs(learned - theory) % TWO_PI
            return np.minimum(diff, TWO_PI - diff)

        return {
            "learned_a": freqs_a,
            "learned_b": freqs_b,
            "learned_a_raw": freqs_a_raw,
            "learned_b_raw": freqs_b_raw,
            "theoretical": theoretical,
            "error_a": _angle_err(freqs_a, theoretical),
            "error_b": _angle_err(freqs_b, theoretical),
        }

pan = PhaseAccumulatorNetwork(p=113, k_freqs=5).to(DEVICE)
print("PAN parameters:", pan.count_parameters())


## The transformer baseline

To compare PAN against something standard, your code uses a one-layer transformer baseline for the same task.

The key point is not that the transformer is wrong. The key point is that it is **generic**. PAN is trying to exploit known structure much more directly.


In [ ]:
class TransformerBaseline(nn.Module):
    def __init__(self, p: int, d_model: int = 128, n_heads: int = 4, d_mlp: int = 512):
        super().__init__()
        self.p = p
        self.d_model = d_model

        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_mlp),
            nn.ReLU(),
            nn.Linear(d_mlp, d_model),
        )
        self.unembed = nn.Linear(d_model, p, bias=False)

        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch = inputs.shape[0]
        eq_token = torch.full((batch, 1), self.p, dtype=torch.long, device=inputs.device)
        seq = torch.cat([inputs, eq_token], dim=1)
        positions = torch.arange(3, device=inputs.device).unsqueeze(0)

        x = self.tok_embed(seq) + self.pos_embed(positions)
        mask = torch.triu(torch.ones(3, 3, device=inputs.device), diagonal=1).bool()
        x_attn, _ = self.attn(x, x, x, attn_mask=mask)
        x = x + x_attn
        x = x + self.mlp(x)
        return self.unembed(x[:, -1, :])

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

tf = TransformerBaseline(p=113).to(DEVICE)
print("Transformer parameters:", tf.count_parameters())


## Training loop

The training loop below mirrors your current implementation closely, including:

- optional `torch.compile`,
- optional validation subsampling,
- early stopping at grokking,
- the **diversity penalty** that discourages all phase-mixing outputs from collapsing onto the same mode.

That diversity penalty matters because otherwise the model can find a degenerate solution where multiple channels stop carrying distinct information.


In [ ]:
def _maybe_compile(model: nn.Module, use_compile: bool) -> nn.Module:
    if not use_compile:
        return model
    if not hasattr(torch, "compile"):
        warnings.warn("torch.compile not available; falling back to eager mode.")
        return model
    try:
        if hasattr(torch, "_dynamo"):
            torch._dynamo.config.suppress_errors = True
            torch._dynamo.config.recompile_limit = 64
        return torch.compile(model, backend="aot_eager")
    except Exception as e:
        warnings.warn(f"torch.compile failed ({e}); falling back to eager mode.")
        return model


def _make_val_loader(val_x, val_y, val_samples: Optional[int]):
    if val_samples is None or val_samples >= len(val_x):
        return val_x, val_y
    idx = torch.randperm(len(val_x), device=val_x.device)[:val_samples]
    return val_x[idx], val_y[idx]


@dataclass
class TrainConfig:
    p: int = 113
    n_steps: int = 50_000
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1.0
    log_every: int = 200
    seed: int = 42
    k_freqs: int = 5
    diversity_weight: float = 0.01
    d_model: int = 128
    n_heads: int = 4
    d_mlp: int = 512
    val_samples: Optional[int] = None
    use_compile: bool = True
    early_stop: bool = True


@dataclass
class TrainHistory:
    steps: list = field(default_factory=list)
    train_loss: list = field(default_factory=list)
    val_loss: list = field(default_factory=list)
    val_acc: list = field(default_factory=list)
    grok_step: Optional[int] = None


def train(model: nn.Module, cfg: TrainConfig, train_x, train_y, val_x, val_y, label: str = "model"):
    torch.manual_seed(cfg.seed)
    model = _maybe_compile(model, cfg.use_compile)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    eval_x, eval_y = _make_val_loader(val_x, val_y, cfg.val_samples)
    history = TrainHistory()
    n_train = len(train_x)
    t0 = time.time()

    for step in range(cfg.n_steps):
        model.train()
        idx = torch.randperm(n_train, device=DEVICE)[:cfg.batch_size]
        x_batch = train_x[idx]
        y_batch = train_y[idx]

        logits = model(x_batch)
        loss = F.cross_entropy(logits, y_batch)

        if cfg.diversity_weight > 0 and hasattr(model, "phase_mix"):
            with torch.no_grad():
                phi_a = model.encoder_a(x_batch[:, 0])
                phi_b = model.encoder_b(x_batch[:, 1])

            mix_out = model.phase_mix(torch.cat([phi_a, phi_b], dim=-1))
            mix_norm = mix_out - mix_out.mean(0, keepdim=True)
            norms = mix_norm.norm(dim=0, keepdim=True).clamp(min=1e-6)
            mix_norm = mix_norm / norms
            gram = mix_norm.T @ mix_norm / mix_out.shape[0]
            eye = torch.eye(gram.shape[0], device=gram.device)
            div_loss = (gram - eye).pow(2).sum() / gram.shape[0]
            loss = loss + cfg.diversity_weight * div_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % cfg.log_every == 0:
            model.eval()
            with torch.no_grad():
                val_logits = model(eval_x)
                val_loss = F.cross_entropy(val_logits, eval_y).item()
                val_acc = (val_logits.argmax(-1) == eval_y).float().mean().item()

            history.steps.append(step)
            history.train_loss.append(loss.item())
            history.val_loss.append(val_loss)
            history.val_acc.append(val_acc)

            if val_acc > 0.99 and history.grok_step is None:
                history.grok_step = step
                print(f"[{label}] grokked at step {step:,} | val_acc={val_acc:.3f} | elapsed={time.time()-t0:.1f}s")
                if cfg.early_stop:
                    break

    return history


## Analysis helpers

The most important mechanistic readouts are:

- whether the learned frequencies stay close to the theoretical Fourier basis,
- whether the decoder becomes Fourier-concentrated,
- whether ablations destroy performance.

Those checks help distinguish “it works” from “it works for the reason we think it works.”


In [ ]:
def fourier_concentration(embed_W: torch.Tensor, top_k: int = 10) -> float:
    W = embed_W.float()
    if W.dim() == 1:
        W = W.unsqueeze(-1)
    W_fft = torch.fft.fft(W, dim=0)
    energy = W_fft.abs() ** 2
    total = energy.sum().item()
    if total < 1e-10:
        return 0.0
    return energy.reshape(-1).topk(top_k).values.sum().item() / total


def analyze_pan(model: PhaseAccumulatorNetwork) -> dict:
    freq_info = model.get_learned_frequencies()

    print("k | theoretical | learned_a | err_a | learned_b | err_b")
    for k in range(model.k_freqs):
        print(
            f"{k+1:>1} | "
            f"{freq_info['theoretical'][k]:>10.6f} | "
            f"{freq_info['learned_a'][k]:>9.6f} | "
            f"{freq_info['error_a'][k]:>7.6f} | "
            f"{freq_info['learned_b'][k]:>9.6f} | "
            f"{freq_info['error_b'][k]:>7.6f}"
        )

    print("\nmean error a:", freq_info["error_a"].mean())
    print("mean error b:", freq_info["error_b"].mean())
    print("SIFP-16 angular quantization:", TWO_PI / PHASE_SCALE)
    return freq_info


def ablation_test(model: nn.Module, val_x, val_y, label: str = "model") -> dict:
    model.eval()
    results = {}

    with torch.no_grad():
        base_logits = model(val_x)
        base_acc = (base_logits.argmax(-1) == val_y).float().mean().item()
        results["baseline"] = base_acc

        if isinstance(model, PhaseAccumulatorNetwork):
            orig_w = model.phase_mix.weight.data.clone()
            model.phase_mix.weight.data.zero_()
            abl = model(val_x)
            results["zero_phase_mixing"] = (abl.argmax(-1) == val_y).float().mean().item()
            model.phase_mix.weight.data.copy_(orig_w)

            orig_fa = model.encoder_a.freq.data.clone()
            orig_fb = model.encoder_b.freq.data.clone()
            model.encoder_a.freq.data = torch.rand_like(orig_fa) * TWO_PI
            model.encoder_b.freq.data = torch.rand_like(orig_fb) * TWO_PI
            abl = model(val_x)
            results["randomize_frequencies"] = (abl.argmax(-1) == val_y).float().mean().item()
            model.encoder_a.freq.data.copy_(orig_fa)
            model.encoder_b.freq.data.copy_(orig_fb)

            orig_ref = model.phase_gate.ref_phase.data.clone()
            model.phase_gate.ref_phase.data.zero_()
            abl = model(val_x)
            results["zero_ref_phases"] = (abl.argmax(-1) == val_y).float().mean().item()
            model.phase_gate.ref_phase.data.copy_(orig_ref)

    print(f"ablation results [{label}]")
    for key, acc in results.items():
        print(f"  {key:<24} {acc:.4f}")
    return results


## A compact experiment you can run

The cell below is set up so it does **not** automatically launch a long run. Flip `RUN_DEMO` to `True` when you want to execute it.

For a fast teaching run on a laptop, try something like:

- `p=67`
- `k_freqs=5`
- `n_steps=10_000`
- `val_samples=1024`

For the fuller benchmark, use your paper settings.


In [ ]:
RUN_DEMO = False

if RUN_DEMO:
    cfg = TrainConfig(
        p=113,
        k_freqs=5,
        n_steps=50_000,
        batch_size=256,
        lr=1e-3,
        weight_decay=0.01,     # better PAN setting than transformer-style 1.0
        diversity_weight=0.01,
        log_every=200,
        seed=42,
        val_samples=1024,
        use_compile=False,
        early_stop=True,
    )

    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)
    pan = PhaseAccumulatorNetwork(cfg.p, cfg.k_freqs).to(DEVICE)
    hist = train(pan, cfg, train_x, train_y, val_x, val_y, label="PAN-demo")

    print("grok step:", hist.grok_step)
    analyze_pan(pan)
    ablation_test(pan, val_x, val_y, label="PAN-demo")


## How to read the result

When PAN works, the most important things to look for are:

1. **accuracy** rises sharply rather than gradually at grokking,
2. **ablations** collapse performance,
3. **learned frequencies** stay close to the Fourier basis or its circular aliases,
4. **parameter count** is dramatically smaller than the transformer baseline.

That does **not** automatically prove the architecture is universally better. It proves something more precise: for this structured task, a phase-native inductive bias is powerful and interpretable.


In [ ]:
def plot_training_history(hist: TrainHistory, title: str = "training history"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist.steps, hist.train_loss, label="train")
    axes[0].plot(hist.steps, hist.val_loss, label="val")
    axes[0].set_title("loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("cross-entropy")
    axes[0].set_yscale("log")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(hist.steps, hist.val_acc, label="val acc")
    if hist.grok_step is not None:
        axes[1].axvline(hist.grok_step, linestyle="--", alpha=0.6, label=f"grok @ {hist.grok_step:,}")
    axes[1].set_title("validation accuracy")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("accuracy")
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Suggested next exercises

A good way to really internalize PAN is to change one thing at a time and predict the outcome before you run it.

Try these:

- remove the diversity penalty and see whether channels collapse,
- change `k_freqs` from 5 to 9 to 12 and compare reliability,
- switch the weight decay from `1.0` to `0.01`,
- inspect the phase-mixing weights after training,
- compare PAN and the transformer at small parameter budgets,
- adapt the task from modular addition to modular multiplication.

That gives you a much stronger feel for which parts are essential and which parts are contingent.
